In [1]:
import time
import os
import requests
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException

In [2]:
# Set up Chrome WebDriver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service)

# Open the BSSDB website
driver.get("https://www.bssdb.dev")

# Maximize the window
driver.maximize_window()

# Wait for the page to load
driver.implicitly_wait(5)

In [3]:
# Create a folder to store images
if not os.path.exists("card_thumbnails"):
    os.makedirs("card_thumbnails")

In [4]:
def extract_data_from_card():
    """
    Extract all relevant card data from the currently loaded web page.
    Returns a JSON object with basic card info, effects, and required cores.
    """
    card_data = {}

    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, "db-bss-zoom-details"))
    )
    
    # Extract basic card data
    card_data["name"] = driver.find_element(By.CSS_SELECTOR, ".db-bss-zoom-details .db-bss-zoom-head h2").text.strip()
    card_data["ID"] = driver.find_element(By.CSS_SELECTOR, ".game-zoom-image").get_attribute("src").split("/")[-1].split(".")[0]
    card_data["set"] = card_data["ID"].split("-")[-2]
    card_data["image"] = f"images/{card_data['set']}/{card_data['ID']}.png"
    
    color_elements = driver.find_elements(By.CSS_SELECTOR, ".segment-title .ui.mini.label")
    card_data["color"] = [
        cls.split()[1].capitalize() for el in color_elements for cls in [el.get_attribute("class")] 
        if any(color in cls for color in ["red", "blue", "green", "yellow", "purple", "white"])
    ]
    
    card_data["cost"] = driver.find_element(By.XPATH, "//div[text()='Cost']/preceding-sibling::div[@class='value']").text
    
    # Attempt to find the element with the "Reduction" label
    reduction_element = driver.find_element(By.XPATH, "//div[text()='Reduction']/preceding-sibling::div[@class='value']")
    # Check if there is text (e.g., a dash for empty reduction)
    reduction_text = reduction_element.text.strip()
    if reduction_text:
        # If there's text, use it directly
        card_data["reduction"] = reduction_text
    else:
        # If there’s no text, look for images (symbols)
        symbols_elements = reduction_element.find_elements(By.XPATH, ".//img")
        card_data["reduction"] = [img.get_attribute('alt').split()[0].lower() for img in symbols_elements]
    
    # Extract symbols
    symbols = driver.find_elements(By.XPATH, "//div[@class='label' and text()='Symbols']/preceding-sibling::div[@class='value']//img")
    card_data["symbols"] = [img.get_attribute('alt').split()[0].lower() for img in symbols]
    
    card_data["cardType"] = driver.find_element(By.XPATH, "//div[text()='Card Type']/preceding-sibling::div[@class='value']").text
    card_data["subType"] = driver.find_element(By.XPATH, "//div[text()='Sub Type(s)']/preceding-sibling::div[@class='value']").text
    card_data["rarity"] = driver.find_element(By.XPATH, "//div[text()='Rarity']/preceding-sibling::div[@class='value']").text
    

    # Extract effects
    effects = []
    effect_elements = driver.find_elements(By.CSS_SELECTOR, ".bss-card-effect")

    for effect in effect_elements:
        effect_data = {}

        # Extract levels
        level_imgs = effect.find_elements(By.CSS_SELECTOR, ".bss-card-effect-level")
        levels = [int(img.get_attribute("alt").replace("LV", "")) for img in level_imgs]
        effect_data["levels"] = levels if levels else []

        # Extract condition
        condition_element = effect.find_elements(By.CSS_SELECTOR, ".bss-card-effect-condition")
        condition = condition_element[0].text.strip() if condition_element else "N/A"
        effect_data["condition"] = condition

        # Extract keywords
        keywords_data = []
        keyword_elements = effect.find_elements(By.CSS_SELECTOR, ".bss-card-effect-keyword")
        for keyword_element in keyword_elements:
            keyword_text = keyword_element.text.strip()
            if ":" in keyword_text:
                keyword, rest = keyword_text.split(": ", 1)
                target, value = rest.rsplit(" x", 1) if " x" in rest else (rest, None)
            else:
                keyword, target, value = keyword_text, None, None

            keywords_data.append({
                "keyword": keyword.strip(),
                "target": target.strip() if target else None,
                "value": int(value) if value else None
            })
        effect_data["keywords"] = keywords_data

        # Extract details
        details_element = effect.find_elements(By.CSS_SELECTOR, ".bss-card-effect-details")
        details = details_element[0].text.strip() if details_element else "N/A"
        effect_data["details"] = details

        # Extract steps (Main, Flash, etc.)
        steps = [step.text.strip() for step in effect.find_elements(By.CSS_SELECTOR, ".bss-card-effect-step")]
        effect_data["steps"] = steps

        effects.append(effect_data)

    # Add extracted effects to card_data
    card_data["effects"] = effects



    # Save the extracted data as JSON
    try:
        folder_path = os.path.join("json", card_data["set"])
        os.makedirs(folder_path, exist_ok=True)
        file_path = os.path.join(folder_path, f"{card_data["ID"]}.json")
        with open(file_path, "w", encoding="utf-8") as json_file:
            json.dump(card_data, json_file, indent=4, ensure_ascii=False)
        print(f"Card data saved to {file_path}")
    except Exception as e:
        print(f"Error saving card data: {e}")

    return card_data

In [5]:
# Function to download the images from the current page
def download_images():
    # Find the image elements based on the `img` tag with `src` containing the thumb URL
    images = driver.find_elements(By.CSS_SELECTOR, "img[src*='thumb']")  # This will match all images whose 'src' contains 'thumb'

    # Download each image and save to the corresponding folder
    for idx, img in enumerate(images):
        # Get the image URL
        img_url = img.get_attribute("src")
        
        # Extract the part of the URL that is used as the image name (e.g., BSS06-001)
        img_name = img_url.split("/")[-1].split(".")[0]  # Split at "/" to get the last part and remove the ".png"
        
        # Get the folder name from the first part of the image name (before the dash)
        folder_name = img_name.split("-")[0]
        
        # Create the folder if it does not exist
        folder_path = f"card_thumbnails/{folder_name}"
        os.makedirs(folder_path, exist_ok=True)
        
        # Download the image
        img_data = requests.get(img_url).content
        
        # Save the image to the folder using the extracted name
        img_path = os.path.join(folder_path, f"{img_name}.png")
        with open(img_path, "wb") as img_file:
            img_file.write(img_data)

In [ ]:
def update_core_requirements():
    """
    Only update the core requirements in the already existing JSON files.
    """
    # Extract the core requirements
    core_requirements = []
    core_elements = driver.find_elements(By.CSS_SELECTOR, ".bss-core-bp")
    
    roman_to_int = {"I": 1, "II": 2, "III": 3}  # Mapping for Roman numerals to integers
    
    for core_element in core_elements:
        level_text = core_element.find_element(By.CSS_SELECTOR, ".level .roman").text.strip()
        battle_points = int(core_element.find_element(By.CSS_SELECTOR, ".bp").text.strip())
        
        # Convert Roman numeral to integer using the mapping
        level_integer = roman_to_int.get(level_text, 0)  # Default to 0 if not found
        
        # Extract the cores value
        cores_value = int(core_element.find_element(By.CSS_SELECTOR, ".cores span").text.strip())
        
        # Create the core data dictionary
        core_data = {
            "level": level_integer,
            "battlePoints": battle_points,
            "cores": cores_value
        }
        
        core_requirements.append(core_data)
    
    # Path for the JSON file (using the card's ID and set as before)
    card_id = driver.find_element(By.CSS_SELECTOR, ".game-zoom-image").get_attribute("src").split("/")[-1].split(".")[0]
    card_set = card_id.split("-")[-2]
    folder_path = os.path.join("..", "json", card_set)
    os.makedirs(folder_path, exist_ok=True)
    file_path = os.path.join(folder_path, f"{card_id}.json")

    # Check if the file exists
    if os.path.exists(file_path):
        # If the file exists, load the existing data
        with open(file_path, "r", encoding="utf-8") as json_file:
            existing_data = json.load(json_file)
        
        # Update only the "core_requirements" field
        existing_data["core_requirements"] = core_requirements

        # Save the updated data back to the JSON file
        with open(file_path, "w", encoding="utf-8") as json_file:
            json.dump(existing_data, json_file, indent=4, ensure_ascii=False)
        print(f"Core requirements updated in {file_path}")
    else:
        print(f"File {file_path} does not exist. Skipping update.")


In [7]:
# Function to cycle through each card on the page
def cycle_through_cards_on_page():
    # Wait for the card images to be visible on the page
    WebDriverWait(driver, 10).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img"))
    )
    
    # Get all card images
    card_images = driver.find_elements(By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")
    
    for index, image in enumerate(card_images):
        # Re-locate the image to avoid stale element issues
        card_images = driver.find_elements(By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")
        image = card_images[index]
        # Scroll into view
        driver.execute_script("arguments[0].scrollIntoView({ behavior: 'smooth', block: 'center' });", image)
        # Wait for the image to be clickable and click it
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")))
        image.click()
        time.sleep(2)  # Wait for the card to open
        # Extract data from the card
        update_core_requirements()
        # Wait for the close button and click it
        close_button = WebDriverWait(driver, 10).until(
            EC.element_to_be_clickable((By.CSS_SELECTOR, "button.bp4-button.bp4-small.bp4-intent-danger"))
        )
        close_button.click()
        time.sleep(1)  # Allow time for the card to close


In [8]:
def navigate_pages():
    # First, cycle through cards on the current page
    cycle_through_cards_on_page()

    # Start by finding the "next page" button and checking its initial state
    next_button = driver.find_element(By.CSS_SELECTOR, "button.bp4-button.bp4-outlined.bp4-intent-primary span.bp4-icon-caret-right")
    
    # Ensure the button is visible and clickable
    while "bp4-disabled" not in next_button.get_attribute("class"):  # This will run while the button is not disabled
        # Scroll the "next" button into view to ensure it's clickable
        driver.execute_script("arguments[0].scrollIntoView();", next_button)
    
        # Wait until the next button is clickable and click it
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable(next_button))
        next_button.click()
    
        # Wait for the page to load by checking for the presence of image elements
        WebDriverWait(driver, 10).until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".db-collection-card-full .game-card-image img")))
    
        # After the page loads, cycle through the cards again
        cycle_through_cards_on_page()
    
        # Optionally, download images if needed
        # download_images()
    
        # Add a slight delay before the next page to avoid overwhelming the server
        time.sleep(2)
    
        # Find the "next page" button again to check if it's still enabled
        next_button = driver.find_element(By.CSS_SELECTOR, "button.bp4-button.bp4-outlined.bp4-intent-primary span.bp4-icon-caret-right")
    
    print("Reached the last page.")



In [9]:
navigate_pages()

Core requirements updated in ..\json\BSS06\BSS06-001.json
Core requirements updated in ..\json\BSS06\BSS06-002.json
Core requirements updated in ..\json\BSS06\BSS06-002_p1.json
Core requirements updated in ..\json\BSS06\BSS06-003.json
Core requirements updated in ..\json\BSS06\BSS06-004.json
Core requirements updated in ..\json\BSS06\BSS06-005.json


KeyboardInterrupt: 

In [ ]:
# Close the browser
driver.quit()